# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR² dataset package](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and record sets from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print summary from dataset metadata
meta = dataset.metadata
print(f"{getattr(meta, 'name', '')}: {getattr(meta, 'description', '')}")
print(f"\nIdentifier: {getattr(meta, 'identifier', '')}")
print(f"Version: {getattr(meta, 'version', '')}")
print(f"Published: {getattr(meta, 'datePublished', '')}")
print(f"License: {getattr(meta, 'license', '')}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` identifiers.
Entities in Croissant are referenced by their unique `@id` attributes for interoperability and clarity.

We will first enumerate available record sets, then their fields/columns.

In [ ]:
import pprint

# Print record set IDs and simple structure
record_set_objs = getattr(meta, 'recordSet', [])
if not record_set_objs:
    print("No record sets found in metadata! (Check Croissant schema under 'recordSet')")
else:
    print("Available record sets and their @id and fields:")
    for rec in record_set_objs:
        rec_id = getattr(rec, '@id', None)
        rec_name = getattr(rec, 'name', None)
        print(f"- Record set: @id='{rec_id}', name='{rec_name}'")
        # List columns or fields for each record set
        fields = getattr(rec, 'field', None)
        if fields:
            if not isinstance(fields, list):
                fields = [fields]
            for f in fields:
                field_id = getattr(f, '@id', None)
                field_name = getattr(f, 'name', None)
                data_type = getattr(f, 'dataType', None)
                print(f"    - Field: @id='{field_id}', name='{field_name}', type='{data_type}'")
        else:
            # Croissant 1.0+ record sets might use 'column' instead
            cols = getattr(rec, 'column', None)
            if cols:
                if not isinstance(cols, list):
                    cols = [cols]
                for c in cols:
                    col_id = getattr(c, '@id', None)
                    col_name = getattr(c, 'name', None)
                    col_type = getattr(c, 'dataType', None)
                    print(f"    - Column: @id='{col_id}', name='{col_name}', type='{col_type}'")
            else:
                print("    (No fields/columns found for this record set)")

### Example: Preview records for a record set
Below, we will preview a few records from each available record set, referencing them by their `@id`.

Replace `<record_set_id>` with the `@id` of a found record set as printed above.

In [ ]:
# List all record set @id's for use below
record_set_ids = [(getattr(rec, '@id', None), getattr(rec, 'name', None)) for rec in getattr(meta, 'recordSet', [])]
pprint.pprint(record_set_ids)

# Print first 3 records for each (show up to 3 record sets)
for i, (record_set_id, record_set_name) in enumerate(record_set_ids[:3]):
    print(f"\nSample from record set '@id': {record_set_id} (name: {record_set_name})")
    try:
        for idx, rec in enumerate(dataset.records(record_set=record_set_id)):
            print(f"Record {idx+1}: {rec}")
            if idx >= 2:
                break
    except Exception as e:
        print(f"  Could not load records: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Use record set and field `@id`s from the previous overview.

We will loop through all available record sets and load them into a dictionary of DataFrames, keyed by record set `@id`.

In [ ]:
# Build a list of record set @id's (from earlier cell)
record_set_ids = [getattr(rec, '@id', None) for rec in getattr(meta, 'recordSet', []) if getattr(rec, '@id', None) is not None]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for '{record_set_id}' (rows: {len(df)}, columns: {list(df.columns)})")
        else:
            dataframes[record_set_id] = pd.DataFrame()
            print(f"No records for '{record_set_id}' (empty DataFrame loaded)")
    except Exception as e:
        print(f"Error loading DataFrame for '{record_set_id}': {e}")

# For further analysis, pick the first non-empty record set
main_record_set_id = None
for rid, df in dataframes.items():
    if df.shape[0] > 0:
        main_record_set_id = rid
        break
if main_record_set_id is None:
    raise ValueError("No non-empty record sets found")

print(f"\nChosen main record set for EDA and analysis: '{main_record_set_id}'")
print("Available columns (@id):", dataframes[main_record_set_id].columns.tolist())
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will select a numeric field (by `@id`) for filtering and normalization, and attempt grouping by another field if available.

In [ ]:
df = dataframes[main_record_set_id]

# Identify numeric fields using DataFrame dtypes
numeric_fields = df.select_dtypes(include='number').columns.tolist()
print(f"Numeric fields discovered: {numeric_fields}")

if not numeric_fields:
    raise ValueError("No numeric fields found in this record set for EDA.")

# Use the first numeric field for demonstration
numeric_field_id = numeric_fields[0]

# Choose a threshold for filtering
threshold = df[numeric_field_id].mean()  # use mean as example cutoff
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} rows\n")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field (preferably not the numeric one)
group_field = None
for c in df.columns:
    if c != numeric_field_id and (df[c].dtype == 'object' or str(df[c].dtype)=='category'):
        group_field = c
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
    print(f"\nGrouped average of '{numeric_field_id}' by '{group_field}' (top 5 groups):")
    display(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we will plot the distribution of our selected numeric field and its grouped means if grouping was performed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
plt.title(f"Distribution of numeric field: {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouped_df exists, plot the means for top categories
if 'grouped_df' in locals() and group_field is not None:
    plt.figure(figsize=(8, 5))
    topn = min(10, grouped_df.shape[0])
    sns.barplot(data=grouped_df.head(topn), x=group_field, y=numeric_field_id)
    plt.xticks(rotation=45, ha='right')
    plt.title(f"Mean of {numeric_field_id} grouped by {group_field} (Top {topn})")
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² dataset package—Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors—using the `mlcroissant` library.

- We loaded dataset metadata and record sets directly from its Croissant schema (`@id` referenced throughout).
- Performed data extraction into DataFrames for further analysis.
- Conducted summary statistics, data filtering, normalization, and grouping on available numeric and categorical fields.
- Visualized distributions and relationships between dataset attributes.

**Next steps**: Continue exploring specific clinical, anatomical, or molecular variables of interest (referencing their `@id`) depending on your analysis objectives. Consider advanced visualizations/correlations or building predictive models for MSI-H status or anatomical characteristics among cancer survivors.
